In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"]
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42

In [ ]:
pyfeat_result_path = "results/fig4_photos_pyfeat_paper.csv"
cluster_result_path = "results/fig3_cluster_results_paper.csv"

In [ ]:
# make label lists with clusters
cluster_df = pd.read_csv(cluster_result_path, dtype={"ID": str})
cluster_df["ID"] = cluster_df["ID"].str.zfill(3)

label_list_list = []
cluster_list = []

for cluster_number, group in cluster_df.groupby("cluster", sort=True):
    label_list = group["ID"].tolist()

    label_list_list.append(label_list)
    cluster_list.append(f"Cluster {int(cluster_number)}")

# read pyfeat results
df = pd.read_csv(pyfeat_result_path, dtype={"label": str})
df["label"] = df["label"].str.zfill(3)

emo_cols = ["anger", "disgust", "fear", "happiness", "sadness", "surprise", "neutral"]

for num, label_list in enumerate(label_list_list):
    filtered_df = df[df["label"].isin(label_list)].copy()

    mean_df = (
        filtered_df.groupby("label")[emo_cols]
        .mean()
        .reindex(label_list)
    )

    # visualize
    heatmap_df = mean_df.T

    if num in [0, 1]:
        plt.figure(figsize=(13, 3))
    elif num==2:
        plt.figure(figsize=(4, 3))
    elif num==3:
        plt.figure(figsize=(3.5, 3))
    elif num==4:
        plt.figure(figsize=(2, 3))
    else:
        plt.figure(figsize=(13, 3))
    im = plt.imshow(
        heatmap_df.values,
        cmap="Reds",
        vmin=0,
        vmax=1,
        aspect="auto"
    )
    
    x_labels = [l[1:] for l in heatmap_df.columns]
    plt.xticks(range(len(heatmap_df.columns)), x_labels)
    if num in [3, 4]:
        plt.yticks([])
    else:
        plt.yticks(range(len(heatmap_df.index)), heatmap_df.index)
    plt.title(cluster_list[num])
    
    for i in range(heatmap_df.shape[0]):
        for j in range(heatmap_df.shape[1]):
            val = heatmap_df.iloc[i, j]
            plt.text(j, i, f"{val:.2f}", ha="center", va="center")
    if num not in [2, 3]:
        plt.colorbar(im)
    plt.tight_layout()
    # save
    plt.savefig(f"results_fig/fig4_paper_c{num+1}.png", dpi=300, bbox_inches="tight", transparent=True)
    plt.show()
    plt.close()